# Chapter 2 - Preprocessing, Pipelines and Hyperparameter Search

**Scenario:** a rental platform wants to predict the monthly rent of a new apartment before the listing is published.

The target is monthly_rent_eur. Keep apartment_rentals_students.csv in the same folder as this notebook.

## 0. Import libraries and dataset

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

df = pd.read_csv("apartment_rentals_students.csv")

# Provided helper: return the main regression metrics for one fitted estimator.
def evaluate_regressor(name, estimator, X, y):
    predictions = estimator.predict(X)
    return {
        "model": name,
        "RMSE": mean_squared_error(y, predictions) ** 0.5,
        "MAE": mean_absolute_error(y, predictions),
        "R2": r2_score(y, predictions),
    }

# Provided helper: convert GridSearchCV results into a compact table.
def search_results_table(search, top_n=10):
    results = pd.DataFrame(search.cv_results_).copy()
    results["CV_RMSE"] = -results["mean_test_score"]
    results["TRAIN_RMSE"] = -results["mean_train_score"]
    results["CV_STD"] = results["std_test_score"]
    columns = [
        "rank_test_score", "CV_RMSE", "CV_STD", "TRAIN_RMSE", "params"
    ]
    return results.sort_values("rank_test_score")[columns].head(top_n)

df.head()

,listing_id,size_m2,num_rooms,floor,distance_to_center_km,building_age_years,furnished,property_type,heating_type,condition,energy_rating,neighborhood,monthly_rent_eur
0,L00001,54.2,2,2,7.9,7,no,apartment,heat_pump,excellent,B,Area_10,1472
1,L00002,100.9,3,2,3.7,41,no,apartment,electric,good,E,Area_02,1847
2,L00003,76.0,3,8,2.8,28,yes,studio,gas,fair,C,Area_15,1770
3,L00004,67.1,2,3,3.9,11,no,apartment,gas,fair,D,Area_07,1559
4,L00005,72.0,2,3,3.7,49,yes,apartment,electric,excellent,B,Area_01,1625


## 1. Understand the dataset

As in the previous practice, the prediction must be made before the apartment is published, using only the information in the listing.

Start by inspecting the data:

In [3]:
print("Shape:", df.shape)

display(
    pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "missing": df.isna().sum(),
    })
)

display(df.describe(include="all").T)

Shape: (1000, 13)


,dtype,n_unique,missing
listing_id,object,1000,0
size_m2,float64,615,0
num_rooms,int64,6,0
floor,int64,10,0
distance_to_center_km,float64,165,0
building_age_years,int64,99,0
furnished,object,2,0
property_type,object,3,0
heating_type,object,4,0
condition,object,4,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
listing_id,1000,1000,L00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size_m2,1000.0,NaN,NaN,NaN,73.3905,27.22726,25.0,54.575,73.9,92.225,156.6
num_rooms,1000.0,NaN,NaN,NaN,2.666,1.174659,1.0,2.0,3.0,3.0,6.0
floor,1000.0,NaN,NaN,NaN,2.941,1.723511,0.0,2.0,3.0,4.0,9.0
distance_to_center_km,1000.0,NaN,NaN,NaN,6.0,3.904962,0.3,3.1,5.1,8.0,20.0
building_age_years,1000.0,NaN,NaN,NaN,34.632,23.409498,0.0,16.75,30.0,48.0,100.0
furnished,1000,2,no,641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
property_type,1000,3,apartment,726,NaN,NaN,NaN,NaN,NaN,NaN,NaN
heating_type,1000,4,gas,355,NaN,NaN,NaN,NaN,NaN,NaN,NaN
condition,1000,4,good,499,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Data dictionary

| Variable | Meaning |
|---|---|
| `listing_id` | Listing identifier |
| `size_m2` | Apartment size in square metres |
| `num_rooms` | Number of rooms |
| `floor` | Floor number |
| `distance_to_center_km` | Distance to the city centre |
| `building_age_years` | Building age |
| `furnished` | Whether the apartment is furnished |
| `property_type` | Studio, apartment or duplex |
| `heating_type` | Main heating system |
| `condition` | Physical condition of the property |
| `energy_rating` | Energy efficiency rating |
| `neighborhood` | Neighborhood name |
| `monthly_rent_eur` | Monthly rent to predict |

## 2. Define feature roles

TODO - Complete the groups below.

```text
Identifier columns:

Target:

Numerical features:

Nominal categorical features:

Ordinal categorical features:

High-cardinality categorical features:
```

In [4]:
# TODO: define the role of each column.

id_columns = [
    # "..."
]

target = "..."

numeric_features = [
    # "..."
]

nominal_features = [
    # "..."
]

ordinal_features = [
    # "..."
]

high_cardinality_features = [
    # "..."
]

# Build X and y only after defining the roles.
X = df.drop(columns=id_columns + [target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

KeyError: "['...'] not found in axis"

TODO - Write a short justification:

```text
The variable(s) that require ordinal encoding are ... because ...

The variable(s) that require one-hot encoding are ... because ...

```

## 3. Apply a train / validation / test split

You already know this split from previous practices. Apply it here **before fitting any encoder or scaler**.

Use approximately:

```text
train       60%
validation  20%
test        20%
```

This is a regression problem, so there is no class label to stratify.

In [ ]:
# TODO: first separate the final test set.
X_temp, X_test, y_temp, y_test = train_test_split(
    ...,
    ...,
    test_size=...,
    random_state=RANDOM_STATE,
)

# TODO: split the remaining data into train and validation.
X_train, X_val, y_train, y_val = train_test_split(
    ...,
    ...,
    test_size=...,
    random_state=RANDOM_STATE,
)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "percentage": [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
})

TODO - Write one sentence:

```text
Encoders and scalers must be fitted after the split because ...
```

## 4. Nominal categorical variables: One-Hot Encoding

With the standard one-hot representation used here, **each observed category becomes its own binary column**.

handle_unknown="ignore" is important because validation or test data may contain a category that was absent from the training set.

In [ ]:
# TODO: create a OneHotEncoder for the low-cardinality nominal variables.
# Keep one output column per category and allow unknown categories at transform time.

nominal_encoder = OneHotEncoder(
    handle_unknown=...,
    drop=...,
    sparse_output=False,
)

# Fit only on the training data.
nominal_encoded = nominal_encoder.fit_transform(X_train[nominal_features])

nominal_encoded_df = pd.DataFrame(
    nominal_encoded,
    columns=nominal_encoder.get_feature_names_out(nominal_features),
    index=X_train.index,
)

nominal_encoded_df.head()

TODO - Write a short interpretation:

```text
The original nominal variables generated ___ encoded columns.

I would not encode property_type as 0, 1, 2 because ...
```

## 5. Ordered categorical variables: Ordinal Encoding

Ordinal variables are different: their categories have a real order. OrdinalEncoder can preserve that order by mapping each category to an ordered numeric value.

Do not let the encoder infer an order alphabetically: define the domain order explicitly.

In [ ]:
# TODO: define the two category orders from worst to best.
condition_order = [...]
energy_order = [...]

# TODO: create the encoder.
ordinal_encoder = OrdinalEncoder(
    categories=[..., ...],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

ordinal_encoded = ordinal_encoder.fit_transform(X_train[ordinal_features])

pd.DataFrame(
    ordinal_encoded,
    columns=ordinal_features,
    index=X_train.index,
).head(10)

TODO - Write a short interpretation:

```text
Ordinal encoding is appropriate here because ...

An arbitrary alphabetical order would be problematic because ...
```

## 6. High-cardinality categorical variables

One-hot encoding is easy to interpret, but a categorical variable with many values can create too many columns.

Inspect neighborhood first.

In [ ]:
neighborhood_counts = X_train["neighborhood"].value_counts()

print("Number of neighborhoods in train:", neighborhood_counts.size)
display(neighborhood_counts.to_frame("count"))

ax = neighborhood_counts.sort_values(ascending=True).plot(
    kind="barh", figsize=(8, 7), legend=False
)
ax.set_xlabel("Training observations")
ax.set_ylabel("Neighborhood")
ax.set_title("Neighborhood frequency in the training set")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: fit a naive one-hot encoder to neighborhood.
naive_neighborhood_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

naive_neighborhood_encoder.fit(X_train[["neighborhood"]])
naive_columns = naive_neighborhood_encoder.get_feature_names_out(["neighborhood"])

# TODO: create a second encoder that groups categories appearing fewer than 20 times.
grouped_neighborhood_encoder = OneHotEncoder(
    handle_unknown=...,
    min_frequency=...,
    sparse_output=False,
)

grouped_neighborhood_encoder.fit(X_train[["neighborhood"]])
grouped_columns = grouped_neighborhood_encoder.get_feature_names_out(["neighborhood"])

pd.DataFrame({
    "encoding": ["one column per category", "group infrequent categories"],
    "output_columns": [naive_columns, grouped_columns],
})

TODO - Write a short interpretation:

```text
Without grouping, neighborhood creates ___ columns.
With min_frequency=20, it creates ___ columns.

Grouping rare categories can help because ...
The trade-off is ...
```

## 7. Numerical variables and scaling

Different models use numerical values differently.

KNN is based on distances. If one variable spans 0–180 and another 0–20, the first can dominate the distance simply because of its units.

Tree-based models do not compare Euclidean distances, so scaling is generally **unnecessary** for Decision Trees and Random Forests.

Inspect the numerical scales in the training set.

In [ ]:
X_train[numeric_features].agg(["min", "max", "mean", "std"]).T

StandardScaler learns a mean and standard deviation **from the training data** and transforms each value so that the column has mean 0 and standard deviation 1.

For KNN we will scale numerical features. For the tree models we will leave them unchanged.

In [ ]:
# TODO: fit StandardScaler only on the numerical training columns.
scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(...)

scaled_numeric_df = pd.DataFrame(
    scaled_numeric,
    columns=numeric_features,
    index=X_train.index,
)

scaled_numeric_df.agg(["mean", "std"]).T

TODO - Write a short interpretation:

```text
Scaling is important for KNN because ...

I would not expect scaling to materially change a Decision Tree because ...
```

## 8. Combine transformations with `ColumnTransformer`

So far each group of columns has been transformed separately for educational purposes. In a real modelling workflow we want a single object that knows what to do with each group.

`ColumnTransformer` applies different transformers to different columns and concatenates the outputs.

We will build **two preprocessing strategies**:

```text
KNN preprocessing
- numerical       -> StandardScaler
- nominal          -> OneHotEncoder
- ordinal          -> OrdinalEncoder + StandardScaler
- neighborhood     -> OneHotEncoder with infrequent-category grouping

Tree preprocessing
- numerical        -> passthrough
- nominal          -> OneHotEncoder
- ordinal          -> OrdinalEncoder
- neighborhood     -> OneHotEncoder with infrequent-category grouping
```

The difference is deliberate: distance-based models need comparable scales; trees do not.

In [ ]:
# TODO: create the preprocessing blocks used by KNN.
knn_numeric = Pipeline([
    ("scaler", ...),
])

knn_ordinal = Pipeline([
    ("encoder", OrdinalEncoder(
        categories=[condition_order, energy_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
    ("scaler", ...),
])

knn_preprocessor = ColumnTransformer([
    ("numeric", ..., numeric_features),
    ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_features),
    ("ordinal", ..., ordinal_features),
    ("high_card", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=20,
    ), high_cardinality_features),
])

# TODO: create the preprocessing used by Decision Trees and Random Forests.
tree_preprocessor = ColumnTransformer([
    ("numeric", ..., numeric_features),
    ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_features),
    ("ordinal", OrdinalEncoder(
        categories=[condition_order, energy_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    ), ordinal_features),
    ("high_card", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=20,
    ), high_cardinality_features),
])

## 9. From preprocessing to a `Pipeline`

A Pipeline chains preprocessing and model fitting into one estimator:

```text
raw X
  ↓
ColumnTransformer
  ↓
model-ready matrix
  ↓
regressor
```

This is not just convenient. During cross-validation, the preprocessing is fitted separately inside every training fold, so validation rows do not influence means, standard deviations, categories or frequency thresholds.

It also lets us tune preprocessing and model hyperparameters together.

In [ ]:
# TODO: build one pipeline for each model family.

knn_pipeline = Pipeline([
    ("preprocessing", ...),
    ("model", KNeighborsRegressor()),
])

tree_pipeline = Pipeline([
    ("preprocessing", ...),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])

rf_pipeline = Pipeline([
    ("preprocessing", ...),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

## 10. Hyperparameter search with cross-validation

Finally, use GridSearchCV on the **training set only**. The validation and test sets remain untouched.

We will optimize RMSE. Because scikit-learn maximizes scores, the scoring name is `neg_root_mean_squared_error`; the helper table converts it back to a positive RMSE.

Pipeline parameters use the syntax:

```text
step__parameter
```

For example:

```text
model__n_neighbors
preprocessing__high_card__min_frequency
```

This means a search can tune both the model and a preprocessing decision.

### 10.1 KNN

Define a small but meaningful grid.

Guidelines:

- n_neighbors: try several values from very local to smoother models; for this dataset, 3–30 is a sensible introductory range.
- weights: include both "uniform" and "distance".
- p: "1" gives Manhattan distance and "2" Euclidean distance.
- also try at least two values for the neighborhood "min_frequency" threshold.

In [ ]:
# TODO: choose candidate values. Do not use an excessively large grid.
knn_param_grid = {
    "model__n_neighbors": [...],
    "model__weights": [...],
    "model__p": [...],
    "preprocessing__high_card__min_frequency": [...],
}

# TODO: configure a 5-fold GridSearchCV using RMSE.
knn_search = GridSearchCV(
    estimator=..., # pipeline prviously declared
    param_grid=...,
    scoring=...,
    cv=...,
    n_jobs=-1,
    return_train_score=True,
)

knn_search.fit(X_train, y_train)
search_results_table(knn_search)

TODO - Interpret the KNN search briefly:

```text
Best hyperparameters:

Best mean CV RMSE:

The values of n_neighbors that performed best suggest that ...

The difference between TRAIN_RMSE and CV_RMSE suggests that ...
```

### 10.2 Decision Tree

Guidelines:

- max_depth: include "None" and several moderate depths.
- min_samples_leaf: include "1" and several larger values.
- the neighborhood grouping threshold can also be part of the search.

Keep the grid compact enough to understand the combinations you are testing.

In [ ]:
# TODO: define candidate values.
tree_param_grid = {
    "model__max_depth": [...],
    "model__min_samples_leaf": [...],
    "preprocessing__high_card__min_frequency": [...],
}

# TODO: configure and run the search.
tree_search = GridSearchCV(
    estimator=...,
    param_grid=...,
    scoring="neg_root_mean_squared_error",
    cv=...,
    n_jobs=-1,
    return_train_score=True,
)

tree_search.fit(X_train, y_train)
search_results_table(tree_search)

TODO - Interpret the Decision Tree search briefly:

```text
Best hyperparameters:

Best mean CV RMSE:

The effect of max_depth / min_samples_leaf appears to be ...

Evidence of overfitting can be seen when ...
```

### 10.3 Random Forest

Guidelines:

- n_estimators: more trees generally stabilize the average, but computation increases. Try 2–3 values rather than dozens.
- max_depth: include "None" and one or two constrained depths.
- min_samples_leaf: compare very small leaves with a more regularized alternative.
- max_features: compare a random subset such as "sqrt" with a larger fraction such as "0.7" or "1.0".
- include at least two "min_frequency" values for the high-cardinality feature.

In [ ]:
# TODO: define a compact Random Forest grid.
rf_param_grid = {
    "model__n_estimators": [...],
    "model__max_depth": [...],
    "model__min_samples_leaf": [...],
    "model__max_features": [...],
    "preprocessing__high_card__min_frequency": [...],
}

# TODO: configure and run the search.
rf_search = GridSearchCV(
    estimator=...,
    param_grid=...,
    scoring="neg_root_mean_squared_error",
    cv=...,
    n_jobs=-1,
    return_train_score=True,
)

rf_search.fit(X_train, y_train)
search_results_table(rf_search)

TODO - Interpret the Random Forest search briefly:

```text
Best hyperparameters:

Best mean CV RMSE:

The hyperparameter(s) that appear to have the largest effect are ...

Changing n_estimators affected performance / stability by ...
```

## 11. Compare the selected pipelines on validation data

Each search has selected one configuration using cross-validation on the training set.

Now evaluate those three fixed configurations on the validation set. Do not tune anything using the test set.

In [ ]:
# TODO: evaluate each best estimator on validation data.
validation_results = pd.DataFrame([
    evaluate_regressor("KNN", ..., X_val, y_val),
    evaluate_regressor("Decision Tree", ..., X_val, y_val),
    evaluate_regressor("Random Forest", ..., X_val, y_val),
]).sort_values("RMSE")

validation_results

TODO - Write a short interpretation:

```text
The model I would select is ...

Its validation RMSE is ... compared with its CV estimate of ...

The agreement / disagreement between CV and validation suggests that ...

I would not choose a model only because its training error is lower because ...
```

## Alternative model-selection workflow

The workflow above uses cross-validation on the training set, validation data to compare the selected model families and the test set for the final evaluation.

An **equally valid alternative** would be to combine the training and validation sets into one development set, use cross-validation on that combined data to select the best model and hyperparameters, and keep the test set untouched until the final performance measurement. This approach allows the model-selection process to use more data, while preserving the test set as an unbiased final evaluation set.